# Runbook 5.5.c: Application security testing (SAST/DAST)

**Skill:** Detect Attacks, Mitigate Risk · **Maps to:** 5.5.B · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Build a miniature SAST rule engine. Given the source lines of a file (as a list of strings, one
per line, 1-indexed), scan each line for known dangerous patterns and return a list of findings, exactly what a real static-analysis tool's rule engine does before a human triages the results.

Detect these categories:
- `"sql-injection-risk"`: a SQL keyword (`SELECT`, `INSERT`, `UPDATE`, `DELETE`) combined with
  string concatenation (`+`) or an f-string/`.format(`/`%` on the same line (i.e., the query is
  being built dynamically instead of parameterized).
- `"eval-use"`: the line calls `eval(` or `exec(`.
- `"hardcoded-secret"`: the line assigns a literal string to a variable whose name contains
  `password`, `secret`, `api_key`, or `token` (case-insensitive), e.g. `password = "hunter2"`.
  An assignment that reads from an environment variable or config object (no literal string on the
  right-hand side) is NOT a finding.

**Lesson objective.** Explain how static application security testing (SAST) and dynamic application security testing (DAST) find different classes of vulnerabilities, describe the tradeoffs of each, and classify a given finding as something SAST would catch, something DAST would catch, or neither.

### What comes in
- `lines`: `list[str]`, source code lines, in file order. Line numbers are 1-indexed (the first
  element of the list is line 1).

### What must come out
A `list[dict]`, each `{"line": int, "category": str}`, **sorted by line number ascending**, then by
category name ascending for findings on the same line. If a line matches more than one category,
emit one finding per matching category. If no lines match, return `[]`.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```
lines = [
  'query = "SELECT * FROM users WHERE name = \'" + user_input + "\'"',
  'password = "hunter2"',
  'eval(user_expr)',
]
solve(lines) -> [
  {"line": 1, "category": "sql-injection-risk"},
  {"line": 2, "category": "hardcoded-secret"},
  {"line": 3, "category": "eval-use"},
]
```

```
lines = ['password = os.environ["DB_PASSWORD"]', 'db.query(stmt, params)']
solve(lines) -> []
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Findings are sorted by `line`, then `category` (both ascending).
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(lines: list) -> list
```

#### Input (`lines`)
`list[str]`, source-code lines, in file order (position 1 = first line, i.e. 1-indexed).

#### Output (`result`)
`list[dict]`, each `{"line": int, "category": str}`, sorted by `line` ascending, then by
`category` ascending for findings on the same line. A single line may contribute more than one
finding if it matches more than one category.

#### Categories
- `"sql-injection-risk"`: a SQL keyword (`SELECT`/`INSERT`/`UPDATE`/`DELETE`, case-insensitive)
  appears on a line that also builds the string dynamically (`+` concatenation, an f-string,
  `.format(`, or `%` formatting) rather than using a parameterized placeholder.
- `"eval-use"`: the line calls `eval(` or `exec(`.
- `"hardcoded-secret"`: the line assigns a quoted string literal to a variable whose name
  contains `password`, `secret`, `api_key`, or `token` (case-insensitive). An assignment that
  reads from an environment variable or config object (no literal string on the right-hand side)
  is NOT a finding.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Parameterized queries (e.g. `execute("... WHERE id = ?", (x))`) are NOT flagged.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 5.5.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""
from typing import Dict, List


def solve(lines: List[str]) -> List[Dict[str, object]]:
    """Scan source lines for known dangerous patterns (miniature SAST rule engine).

    Args:
        lines: list of source-code line strings, 1-indexed by position.
    Returns:
        list of {"line": int, "category": str} findings, sorted by line then category
        (both ascending); a line may contribute more than one finding. see the problem statement above for
        the exact category rules.
    """
    # TODO: implement per the problem statement above
    raise NotImplementedError("implement solve() — see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

import os


def test_mixed_findings():
    lines = [
        'query = "SELECT * FROM users WHERE name = \'" + user_input + "\'"',
        'password = "hunter2"',
        'eval(user_expr)',
    ]
    assert impl.solve(lines) == [
        {"line": 1, "category": "sql-injection-risk"},
        {"line": 2, "category": "hardcoded-secret"},
        {"line": 3, "category": "eval-use"},
    ]


def test_safe_lines_no_findings():
    lines = ['password = os.environ["DB_PASSWORD"]', 'db.query(stmt, params)']
    assert impl.solve(lines) == []


def test_clean_code_no_findings():
    lines = ["def add(a, b):", "    return a + b", "", "result = add(1, 2)"]
    assert impl.solve(lines) == []


def test_parameterized_sql_not_flagged():
    lines = ['cur.execute("SELECT * FROM users WHERE id = ?", (user_id,))']
    assert impl.solve(lines) == []


def test_exec_flagged_same_as_eval():
    lines = ["exec(payload)"]
    assert impl.solve(lines) == [{"line": 1, "category": "eval-use"}]


def test_multiple_categories_same_line():
    lines = ['token = "abc123"; eval(token)']
    result = impl.solve(lines)
    assert result == [
        {"line": 1, "category": "eval-use"},
        {"line": 1, "category": "hardcoded-secret"},
    ]


def test_case_insensitive_secret_keyword_and_sql():
    lines = ['API_KEY = "sk_live_xyz"', 'q = "select * from t where x=" + v']
    assert impl.solve(lines) == [
        {"line": 1, "category": "hardcoded-secret"},
        {"line": 2, "category": "sql-injection-risk"},
    ]


def test_fstring_sql_flagged():
    lines = ['q = f"SELECT * FROM t WHERE id={uid}"']
    assert impl.solve(lines) == [{"line": 1, "category": "sql-injection-risk"}]


def test_empty_input():
    assert impl.solve([]) == []


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([]),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it